# E3 — 04 — CNN + atención

Experimento E2 con la CNN: ¿ayuda ajustar los embeddings de eswiki a nuestra tarea, o es mejor dejarlos congelados? Se hace para las dos tareas, detección y tipo.

- Se entrena con el train de los tres corpus mezclados y se elige (Optuna y parada temprana) con el **dev de Political**. El test no se mira aquí: las métricas finales salen de `06_comparativa`.
- La atención y el entrenamiento vienen de `redes_comun.py`, el mismo código que usa la BiLSTM. Así, si una red gana a la otra, es por su bloque central y no por cómo se ha entrenado.
- Cada variante se entrena 3 veces (semillas 0, 1 y 2): la red cambia de un entrenamiento a otro (pesos iniciales, orden de los lotes, dropout).

| Salida | Contenido |
|---|---|
| `predicciones/{tarea}__cnn__{variante}__s{semilla}.csv` | Predicciones de dev y test (`id`, `split`, `real`, `predicho`) |
| `mejores_params_cnn.json` | Mejores hiperparámetros de cada tarea |

E3 (solo Political frente a mezclado) y E3b (preentreno) se harán después, con el embedding que gane aquí.

## 1. Datos

Todo viene de `data/e3/` (notebook `01_datos`):
- `secuencias.npy`: cada texto como 150 índices de palabra (0 = relleno, 1 = desconocida).
- `matriz_eswiki.npy`: el vector de eswiki de cada índice, para inicializar el embedding.

La fila *i* de `datos_E3.jsonl` es la fila *i* de `secuencias.npy`, así que el índice de la tabla sirve para sacar las secuencias. Las etiquetas del tipo se pasan a número con el orden de `clases_tipo.json`.

In [1]:
import json
import os
import time

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn

from redes_comun import Atencion, fijar_semilla, entrenar, predecir

CARPETA = "../data/e3"

df = pd.read_json(f"{CARPETA}/datos_E3.jsonl", lines=True)
secuencias = torch.tensor(np.load(f"{CARPETA}/secuencias.npy"), dtype=torch.long)
matriz_eswiki = torch.tensor(np.load(f"{CARPETA}/matriz_eswiki.npy"))
with open(f"{CARPETA}/clases_tipo.json", encoding="utf-8") as f:
    clases_tipo = json.load(f)
with open(f"{CARPETA}/vocabulario_secuencias.json", encoding="utf-8") as f:
    indice_a_palabra = {i: p for p, i in json.load(f).items()}

print(len(df), secuencias.shape, matriz_eswiki.shape)

C:\Users\asier.gomez\Proyectos\Clase\detector_falacias_contradicciones\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


7843 torch.Size([7843, 150]) torch.Size([18437, 300])


## 2. Las dos tareas

- **Detección:** todos los textos, 2 clases (0 sin falacia, 1 falacia).
- **Tipo:** solo los textos con falacia, 11 clases. `explode` duplica en train los 9 textos con dos etiquetas; en dev y test todos tienen una sola, así que solo la saca de la lista.

`separar` devuelve el train (los tres corpus) y el dev de Political, que es con lo que se elige.

In [2]:
tipo = df[df["etiqueta_tipo"].notna()].explode("etiqueta_tipo")

TAREAS = {
    "deteccion": {"tabla": df.assign(etiqueta=df["etiqueta_deteccion"]), "clases": [0, 1]},
    "tipo": {"tabla": tipo.assign(etiqueta=tipo["etiqueta_tipo"].map(clases_tipo.index)), "clases": clases_tipo},
}


def separar(tabla):
    train = tabla[tabla["split"] == "train"]
    dev = tabla[(tabla["split"] == "dev") & (tabla["corpus"] == "political")]
    return (secuencias[train.index], torch.tensor(train["etiqueta"].values),
            secuencias[dev.index], torch.tensor(dev["etiqueta"].values))


for nombre, tarea in TAREAS.items():
    X_tr, y_tr, X_dv, y_dv = separar(tarea["tabla"])
    print(nombre, "train:", len(y_tr), " dev Political:", len(y_dv))

deteccion train: 5512  dev Political: 401
tipo train: 3200  dev Political: 289


## 3. La red

```
texto: 150 índices
  → embedding de eswiki (300 por palabra; congelado o ajustado)
  → 3 convoluciones en paralelo, con ventanas de 3, 5 y 7 palabras y `tamano` filtros cada una
  → 150 posiciones × 3·tamano (cada posición resume la frase centrada en esa palabra)
  → atención: un peso por posición (el relleno no cuenta) y media ponderada
  → dropout → capa lineal → 2 u 11 clases
```

- Cada **filtro** es un detector de patrones que recorre el texto y da un número en cada posición: cuánto se parecen las palabras de su ventana al patrón que ha aprendido. Hay tres tamaños de ventana porque hay patrones de distinta longitud.
- `padding="same"` añade ceros por los lados para que la salida tenga también 150 posiciones, una por palabra; con ventanas impares quedan centradas. Así la máscara del relleno sigue cuadrando con la atención.
- La **atención** sustituye al max-pooling de la CNN clásica (Kim, 2014): en vez de quedarse con el máximo de cada filtro, aprende qué zonas del texto importan y las usa enteras. Sus pesos se pueden dibujar.
- `padding_idx=0`: el vector del relleno queda a cero y no se ajusta.
- Una sola capa convolucional: con unos pocos miles de textos de train, más capas sobreajustarían, y el contexto largo ya lo recoge la atención.

In [3]:
VENTANAS = (3, 5, 7)


class CNN(nn.Module):
    def __init__(self, tamano, dropout, congelar, n_clases):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(matriz_eswiki, freeze=congelar, padding_idx=0)
        self.convoluciones = nn.ModuleList([nn.Conv1d(300, tamano, k, padding="same") for k in VENTANAS])
        self.atencion = Atencion(len(VENTANAS) * tamano)
        self.dropout = nn.Dropout(dropout)
        self.salida = nn.Linear(len(VENTANAS) * tamano, n_clases)

    def forward(self, x):
        mascara = x != 0                                   # 0 = relleno
        e = self.embedding(x).transpose(1, 2)              # Conv1d quiere (lote, 300, palabras)
        h = torch.cat([torch.relu(conv(e)) for conv in self.convoluciones], dim=1)
        vector, pesos = self.atencion(h.transpose(1, 2), mascara)
        return self.salida(self.dropout(vector)), pesos

## 4. Búsqueda de hiperparámetros (Optuna)

Una búsqueda por tarea, **solo con eswiki congelado**. Sus hiperparámetros se reutilizan en la variante ajustada: la entrada y la red son las mismas, así que la diferencia del E2 se deberá al embedding.

Cada intento crea la red con la semilla 0, la entrena (`entrenar`, con parada temprana) y se puntúa con el F1 macro del dev de Political. Se busca:
- `lr` (0,0001–0,01, escala logarítmica): el tamaño de cada paso de Adam.
- `dropout` (0,1–0,5): la proporción de valores que se apagan al azar al entrenar, para no sobreajustar.
- `tamano` (50–200): nº de filtros por ventana.
- `batch_size` (32, 64 o 128): textos por lote.

Sampler TPE (`seed=42`), como en el SVM. Con **poda** (`MedianPruner`): tras cada época, el intento informa de su F1 y se corta si va peor que la mediana de los intentos anteriores en esa misma época. No corta en los 5 primeros intentos ni en las 2 primeras épocas, para tener con qué comparar.

In [4]:
N_INTENTOS = 20
RUTA_PARAMS = "mejores_params_cnn.json"
optuna.logging.set_verbosity(optuna.logging.WARNING)


def buscar(nombre_tarea):
    tarea = TAREAS[nombre_tarea]
    X_tr, y_tr, X_dv, y_dv = separar(tarea["tabla"])
    n_clases = len(tarea["clases"])

    def intento(trial):
        params = {
            "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
            "dropout": trial.suggest_float("dropout", 0.1, 0.5),
            "tamano": trial.suggest_int("tamano", 50, 200),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
        }
        fijar_semilla(0)
        red = CNN(params["tamano"], params["dropout"], congelar=True, n_clases=n_clases)
        return entrenar(red, X_tr, y_tr, X_dv, y_dv, params["lr"], params["batch_size"], n_clases, trial=trial)

    estudio = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
    )
    inicio = time.time()
    estudio.optimize(intento, n_trials=N_INTENTOS)
    podados = sum(t.state == optuna.trial.TrialState.PRUNED for t in estudio.trials)
    print(f"{nombre_tarea}: mejor F1 {estudio.best_value:.3f}, {podados} intentos podados, "
          f"{(time.time() - inicio) / 60:.0f} min")
    guardar_params(nombre_tarea, estudio.best_params)
    return estudio.best_params


def guardar_params(clave, params):
    # se lee y se reescribe, para no borrar la otra tarea si se ejecuta una sola
    todos = {}
    if os.path.exists(RUTA_PARAMS):
        with open(RUTA_PARAMS) as f:
            todos = json.load(f)
    todos[clave] = params
    with open(RUTA_PARAMS, "w") as f:
        json.dump(todos, f, indent=2)

## 4.1. Búsqueda para la detección

In [5]:
params_deteccion = buscar("deteccion")
params_deteccion

deteccion: mejor F1 0.721, 5 intentos podados, 24 min


{'lr': 0.0024503913359360924,
 'dropout': 0.23682146109987837,
 'tamano': 96,
 'batch_size': 64}

## 4.2. Búsqueda para el tipo

In [6]:
params_tipo = buscar("tipo")
params_tipo

tipo: mejor F1 0.293, 15 intentos podados, 12 min


{'lr': 0.004622589001020831,
 'dropout': 0.18493564427131048,
 'tamano': 77,
 'batch_size': 128}

## 5. E2: eswiki congelado frente a ajustado

Con los hiperparámetros de su tarea, cada variante se entrena 3 veces (semillas 0, 1 y 2) y se guardan sus predicciones:
- `eswiki_congelado`: los vectores de eswiki no cambian; solo aprenden la CNN, la atención y la capa final.
- `eswiki_ajustado`: los vectores también se ajustan a nuestra tarea (5,5 millones de números más que aprender).

El modelo de la semilla 0 se guarda en memoria (no en disco) para los ejemplos de atención de la sección 7.

In [7]:
PARAMS = {"deteccion": params_deteccion, "tipo": params_tipo}
modelos = {}


def guardar_predicciones(red, nombre_tarea, variante, semilla):
    tarea = TAREAS[nombre_tarea]
    filas = tarea["tabla"][tarea["tabla"]["split"].isin(["dev", "test"])]
    clases = tarea["clases"]
    salida = pd.DataFrame({
        "id": filas["id"],
        "split": filas["split"],
        "real": [clases[i] for i in filas["etiqueta"]],
        "predicho": [clases[i] for i in predecir(red, secuencias[filas.index])],
    })
    os.makedirs("predicciones", exist_ok=True)
    salida.to_csv(f"predicciones/{nombre_tarea}__cnn__{variante}__s{semilla}.csv", index=False)


def e2(nombre_tarea, variante):
    tarea, p = TAREAS[nombre_tarea], PARAMS[nombre_tarea]
    X_tr, y_tr, X_dv, y_dv = separar(tarea["tabla"])
    n_clases = len(tarea["clases"])

    f1s = []
    for semilla in (0, 1, 2):
        fijar_semilla(semilla)
        red = CNN(p["tamano"], p["dropout"], congelar=(variante == "eswiki_congelado"), n_clases=n_clases)
        f1s.append(entrenar(red, X_tr, y_tr, X_dv, y_dv, p["lr"], p["batch_size"], n_clases))
        guardar_predicciones(red, nombre_tarea, variante, semilla)
        if semilla == 0:
            modelos[(nombre_tarea, variante)] = red

    return {"tarea": nombre_tarea, "variante": variante,
            "F1 macro dev Political": f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
            "por semilla": np.round(f1s, 3).tolist()}

## 5.1. Detección con eswiki congelado

In [8]:
r1 = e2("deteccion", "eswiki_congelado")
r1

{'tarea': 'deteccion',
 'variante': 'eswiki_congelado',
 'F1 macro dev Political': '0.714 ± 0.005',
 'por semilla': [0.721, 0.712, 0.709]}

## 5.2. Detección con eswiki ajustado

In [9]:
r2 = e2("deteccion", "eswiki_ajustado")
r2

{'tarea': 'deteccion',
 'variante': 'eswiki_ajustado',
 'F1 macro dev Political': '0.701 ± 0.015',
 'por semilla': [0.718, 0.703, 0.682]}

## 5.3. Tipo con eswiki congelado

In [10]:
r3 = e2("tipo", "eswiki_congelado")
r3

{'tarea': 'tipo',
 'variante': 'eswiki_congelado',
 'F1 macro dev Political': '0.302 ± 0.008',
 'por semilla': [0.31, 0.291, 0.305]}

## 5.4. Tipo con eswiki ajustado

In [11]:
r4 = e2("tipo", "eswiki_ajustado")
r4

{'tarea': 'tipo',
 'variante': 'eswiki_ajustado',
 'F1 macro dev Political': '0.317 ± 0.014',
 'por semilla': [0.335, 0.316, 0.3]}

## 6. Resumen

F1 macro en el dev de Political de cada configuración (media ± desviación de las 3 semillas). Sirve para elegir el embedding que se usará en E3 y E3b; la comparación con los otros modelos y las métricas del test se hacen en `06_comparativa`.

In [12]:
pd.DataFrame([r1, r2, r3, r4])

,tarea,variante,F1 macro dev Political,por semilla
0,deteccion,eswiki_congelado,0.714 ± 0.005,"[0.721, 0.712, 0.709]"
1,deteccion,eswiki_ajustado,0.701 ± 0.015,"[0.718, 0.703, 0.682]"
2,tipo,eswiki_congelado,0.302 ± 0.008,"[0.31, 0.291, 0.305]"
3,tipo,eswiki_ajustado,0.317 ± 0.014,"[0.335, 0.316, 0.3]"


## 7. Ejemplos de atención

Para la mejor variante de cada tarea (modelo de la semilla 0), cinco textos del dev de Political que el modelo acierta, con las cinco palabras que más peso reciben. Como cada posición resume la frase centrada en esa palabra, un peso alto señala esa zona del texto.

Los pesos dicen dónde mira el modelo, no exactamente por qué decide lo que decide (Jain y Wallace, 2019).

In [13]:
def ejemplos_atencion(nombre_tarea, variante, n_textos=5, n_palabras=5):
    red, tarea = modelos[(nombre_tarea, variante)], TAREAS[nombre_tarea]
    dev = tarea["tabla"][(tarea["tabla"]["split"] == "dev") & (tarea["tabla"]["corpus"] == "political")]
    X = secuencias[dev.index]
    red.eval()
    with torch.no_grad():
        logits, pesos = red(X)
    aciertos = np.where(logits.argmax(dim=1).numpy() == dev["etiqueta"].values)[0][:n_textos]

    print(f"=== {nombre_tarea}, {variante} ===")
    for i in aciertos:
        mas_peso = torch.argsort(pesos[i], descending=True)[:n_palabras]
        palabras = [f"{indice_a_palabra[int(X[i, j])]} ({pesos[i, j]:.2f})" for j in mas_peso]
        print(f"[{tarea['clases'][dev['etiqueta'].values[i]]}] {dev['texto'].values[i][:150]}")
        print("   ->", ", ".join(palabras))


def mejor_variante(nombre_tarea):
    filas = [r for r in (r1, r2, r3, r4) if r["tarea"] == nombre_tarea]
    return max(filas, key=lambda r: np.mean(r["por semilla"]))["variante"]


for nombre_tarea in TAREAS:
    ejemplos_atencion(nombre_tarea, mejor_variante(nombre_tarea))
    print()

=== deteccion, eswiki_congelado ===
[1] Yo sí me comprometo, y soy el único candidato que lo ha hecho y lo va a hacer seguramente hoy, a desbloquear el país desde la oposición para hablar de
   -> país (0.10), desbloquear (0.08), hablar (0.07), de (0.07), y (0.06)
[1] aunque haya cinco candidatos, solo hay dos posibilidades para ganar esas elecciones.
   -> aunque (0.27), ganar (0.13), para (0.12), elecciones (0.07), esas (0.07)
[1] El señor Sánchez dice que en Cataluña no pasa nada, que hay normalidad. ¿Le parece normal que vuelen adoquines en la Plaza Urquinaona, en el Paseo de 
   -> no (0.11), solucionan (0.11), la (0.08), la (0.07), problemas (0.06)
[1] Antes de dar lecciones, señor Casado, yo podría decirle, por ejemplo, que quien transfirió las competencias de tráfico de justicia, de seguridad, de p
   -> , (0.10), una (0.08), quien (0.05), . (0.04), . (0.04)
[1] A esos que hoy Torra les llama bestias taradas, esos eran mis abuelos, esos eran mis padres y esos son millones de ca